# Giai đoạn 3 & 4: Bài toán Machine Learning (ML Question)

**Môn học**: CSC17104 – Programming for Data Science  
**Đồ án**: Final Project  
**Nhóm sinh viên**: KdF (Lê Thanh Hải - 23127360, Ngô Quang Thắng - 23127473)  
**Người phụ trách chính**: Ngô Quang Thắng (cả hai thành viên cùng nắm rõ 100%)

---

> **Nguyên tắc cốt lõi của môn học**:  
> Mô hình Machine Learning là **phương tiện** để trả lời một câu hỏi thực tế có ý nghĩa, không phải là đích đến cuối cùng.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, classification_report
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance

# Cấu hình hiển thị
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Định nghĩa bài toán dự đoán (Prediction Task Definition)

- **Câu hỏi nghiên cứu ML**: [Phát biểu câu hỏi nghiên cứu cụ thể, ví dụ: Dự đoán điểm đánh giá trung bình `Rating Average` hoặc phân loại độ phổ biến của board game dựa trên các đặc tính thiết kế].
- **Người dùng mục tiêu (Clear user)**: [Ai là người hưởng lợi từ mô hình này? Ví dụ: Nhà phát triển game độc lập muốn ước lượng phản hồi cộng đồng trước khi xuất bản].
- **Quyết định hỗ trợ**: [Mô hình giúp họ ra quyết định gì về mặt thiết kế/kinh doanh?].
- **Tác vụ dự đoán**: [Hồi quy (Regression) / Phân loại (Classification)].
- **Phòng chống rò rỉ dữ liệu (Avoid Data Leakage)**:
  - Liệt kê rõ ràng danh sách các cột **bị loại bỏ** khỏi tập đặc trưng vì chúng không có sẵn tại thời điểm dự đoán thực tế (ví dụ: `BGG Rank`, `Users Rated`, `Owned Users` là các chỉ số hậu phát hành, nếu đưa vào sẽ gây data leakage nghiêm trọng).

## 2. Chiến lược chia dữ liệu (Data Splitting Strategy)

- **Phương pháp chia tập dữ liệu**:
  - Giải thích lựa chọn phương pháp chia phù hợp: Time-based split (chia theo năm phát hành `Year Published` để mô phỏng tương lai) hoặc Group-based / Stratified split.
- **Tỷ lệ Train / Test**: (Ví dụ 80% Train, 20% Test).
- **Quy tắc đóng băng Test Set**: Tập Test được tách độc lập và đóng băng hoàn toàn ngay từ đầu, chỉ được đánh giá duy nhất 1 lần ở bước cuối cùng.

In [ ]:
# Code thực hiện chia tập Train và Test độc lập
# X = df.drop(columns=['Rating Average', ...])
# y = df['Rating Average']
# X_train, X_test, y_train, y_test = train_test_split(...)
# print(f"Kích thước Train: {X_train.shape}, Test: {X_test.shape}")

## 3. Xây dựng Pipeline & So sánh mô hình (Baseline vs Candidate Models)

- **Mô hình cơ sở (Trivial baseline)**: Thiết lập baseline tầm thường (dự đoán giá trị trung bình `DummyRegressor(strategy='mean')` hoặc nhãn đa số `DummyClassifier(strategy='most_frequent')`). Mọi mô hình phức tạp đều phải vượt qua baseline này một cách thuyết phục.
- **Quy chuẩn Pipeline bắt buộc**: Toàn bộ các bước tiền xử lý (điền khuyết thiếu bằng `SimpleImputer`, chuẩn hóa thang đo bằng `StandardScaler`, mã hóa phân loại bằng `OneHotEncoder`) bắt buộc phải đóng gói bên trong `scikit-learn Pipeline` và `ColumnTransformer` để ngăn chặn data leakage giữa tập train và tập test.
- **Các mô hình ứng viên (Candidate models)**: Xây dựng và so sánh ít nhất 2 mô hình máy học khác nhau (ví dụ: Linear/Ridge Model vs Random Forest / Gradient Boosting).

In [ ]:
# Định nghĩa tiền xử lý cho biến số và biến phân loại
# numeric_features = [...]
# categorical_features = [...]

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
#         ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
#     ]
# )

# Xây dựng các Pipeline cho Baseline, Model 1, Model 2

## 4. Tinh chỉnh siêu tham số (Hyperparameter Tuning) & Đánh giá trên Test Set

- **Cross-Validation**: Sử dụng K-Fold Cross-Validation CHỈ TRÊN TẬP TRAIN (kết hợp với `GridSearchCV` hoặc `RandomizedSearchCV`) để tìm bộ siêu tham số tối ưu.
- **Đánh giá trên Test Set**: Sau khi chọn được mô hình tốt nhất từ cross-validation, đánh giá mô hình đó trên tập Test ĐÚNG 1 LẦN DUY NHẤT.
- **Biện minh cho Metric đánh giá**: Giải thích rõ lý do lựa chọn metric (ví dụ: tại sao chọn MAE/RMSE cho bài toán hồi quy, hoặc tại sao không dùng Accuracy khi tập dữ liệu mất cân bằng trong bài toán phân loại).

In [ ]:
# Code thực hiện Cross-Validation Tuning trên Train Set và đánh giá 1 lần trên Test Set

## 5. Phân tích lỗi (Error Analysis)

Mô hình không hoàn hảo. Mục tiêu là tìm hiểu xem mô hình dự đoán kém hoặc thất bại ở đâu:
- Phân tích sai số theo các nhóm đối tượng cụ thể (ví dụ: game chiến thuật nặng vs game trẻ em, game phát hành trước năm 2000 vs game hiện đại).
- Biểu đồ phân phối phần dư (Residual distribution plot) và sai số theo dải giá trị thực tế.
- Trích xuất và phân tích cụ thể 3–5 tựa game mà mô hình dự đoán sai lệch nhiều nhất (Worst prediction examples): Tìm hiểu nguyên nhân vì sao mô hình lại đoán lệch như vậy.

In [ ]:
# Code phân tích sai số và trích xuất các mẫu dự đoán sai nhiều nhất

## 6. Diễn giải mô hình & Trả lời câu hỏi nghiên cứu (Interpretation & Conclusion)

- **Kỹ thuật diễn giải mô hình**: Sử dụng `permutation_importance` (hoặc hệ số hồi quy / feature importances) để xác định xem đặc trưng nào đóng vai trò quyết định nhất đến kết quả dự đoán.
- **Biểu đồ độ quan trọng đặc trưng**: Trực quan hóa top 10–15 đặc trưng có ảnh hưởng mạnh nhất.
- **Trả lời câu hỏi nghiên cứu ban đầu**: Đúc kết câu trả lời thực tế dựa trên những gì mô hình đã học (ví dụ: thể loại hay độ phức tạp quyết định điểm rating nhiều hơn? Những cơ chế nào mang lại điểm đánh giá cao cho một tựa game?).

In [ ]:
# Code trực quan hóa Permutation Importance và diễn giải mô hình

## 7. Tài liệu tham khảo (References)
- scikit-learn Documentation (Pipelines, ColumnTransformer, Permutation Importance): https://scikit-learn.org/
- BoardGameGeek Database: https://boardgamegeek.com/